# Example Dirichlet Surface

Canonical Dirichlet family notebook for real interval zeta/eta and complex-box `acb_dirichlet` point/basic surfaces.

## Scope

This notebook is the canonical example surface for `example_dirichlet_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import io
import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_dirichlet_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_dirichlet_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view. Canonical retained execution in this repo state is CPU-oriented, but the notebook calling pattern remains CPU/GPU portable and explicitly parameterized for `float32` and `float64`.

In [ ]:
SUPPORTED_JAX_MODES = ('cpu', 'gpu')
SUPPORTED_JAX_DTYPES = ('float32', 'float64')
if JAX_MODE not in SUPPORTED_JAX_MODES:
    raise ValueError(f'Unsupported JAX_MODE: {JAX_MODE}')
if JAX_DTYPE not in SUPPORTED_JAX_DTYPES:
    raise ValueError(f'Unsupported JAX_DTYPE: {JAX_DTYPE}')
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('supported_jax_modes:', SUPPORTED_JAX_MODES)
print('supported_jax_dtypes:', SUPPORTED_JAX_DTYPES)
print('validation_slice:', 'cpu_current__gpu_portable_contract')
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)
runtime_payload = json.loads(runtime.stdout)
(EXAMPLE_OUTPUT_ROOT / f'runtime_{JAX_MODE}.json').write_text(json.dumps(runtime_payload, indent=2) + '\n', encoding='utf-8')

## Direct Usage

Construct representative real intervals and complex boxes, then evaluate Dirichlet zeta and eta on both the real and complex helper surfaces.

In [ ]:
import jax.numpy as jnp
from arbplusjax import acb_core, acb_dirichlet, api, dirichlet, double_interval as di

s_real = di.interval(jnp.array([1.5, 2.0, 2.5], dtype=jnp.float64), jnp.array([1.55, 2.05, 2.55], dtype=jnp.float64))
s_complex = acb_core.acb_box(
    di.interval(jnp.array([1.5, 2.0, 2.5], dtype=jnp.float64), jnp.array([1.55, 2.05, 2.55], dtype=jnp.float64)),
    di.interval(jnp.array([-0.2, -0.05, 0.1], dtype=jnp.float64), jnp.array([-0.15, 0.0, 0.15], dtype=jnp.float64)),
)
direct_results = {
    'dirichlet_zeta': dirichlet.dirichlet_zeta_batch(s_real, n_terms=32),
    'dirichlet_eta': dirichlet.dirichlet_eta_batch(s_real, n_terms=32),
    'acb_dirichlet_zeta': acb_dirichlet.acb_dirichlet_zeta_batch(s_complex, n_terms=48),
    'acb_dirichlet_eta': acb_dirichlet.acb_dirichlet_eta_batch(s_complex, n_terms=48),
}
display(direct_results)

## Production Pattern

For repeated Dirichlet usage, keep `n_terms`, `dtype`, and `pad_to` stable. Use the real batch JIT path directly for interval zeta/eta, and use the API-bound compiled point batch for the complex `acb_dirichlet` surfaces.

In [ ]:
real_zeta = dirichlet.dirichlet_zeta_batch_jit(s_real, n_terms=32)
real_eta = dirichlet.dirichlet_eta_batch_jit(s_real, n_terms=32)
real_zeta_basic = dirichlet.dirichlet_zeta_batch_prec_jit(s_real, n_terms=32, prec_bits=53)
complex_zeta_bound = api.bind_point_batch_jit('acb_dirichlet_zeta', dtype='float64', pad_to=8, n_terms=48)
complex_eta_bound = api.bind_point_batch_jit('acb_dirichlet_eta', dtype='float64', pad_to=8, n_terms=48)
service_results = {
    'real_zeta_jit': real_zeta,
    'real_eta_jit': real_eta,
    'real_zeta_basic': real_zeta_basic,
    'complex_zeta_bound': complex_zeta_bound(s_complex),
    'complex_eta_bound': complex_eta_bound(s_complex),
}
display(service_results)

## Extending Benchmarks

To extend Dirichlet benchmarks, add stable metric rows to `benchmark_dirichlet.py` and `benchmark_acb_dirichlet.py`, and keep the stdout summary format stable for notebook parsing.

## Fast JAX Point Pattern

The complex Dirichlet helper family already has a compiled point-batch API path. Keep `n_terms` static and `pad_to` fixed so repeated compiled calls stay on the same kernel.

In [ ]:
import jax
jit_bound = api.bind_point_batch_jit('acb_dirichlet_zeta', dtype='float64', pad_to=8, n_terms=48)
jit_vals = jit_bound(s_complex)
vmap_vals = jax.vmap(lambda s_i: api.eval_point('acb_dirichlet_zeta', s_i, n_terms=48))(s_complex)
display({'jit_shape': jit_vals.shape, 'jit_matches_vmap': bool(jnp.allclose(jit_vals, vmap_vals, rtol=1e-10, atol=1e-10))})

## AD Product Pattern

AD should be shown on the actual Dirichlet public surfaces. This section differentiates the real interval midpoint path and the real part of the complex-box zeta path, then plots primal and gradient values.

In [ ]:
import jax
sweep = jnp.linspace(1.3, 2.7, 24, dtype=jnp.float64)
real_primal = jax.vmap(lambda x: jnp.squeeze(di.midpoint(dirichlet.dirichlet_zeta(di.interval(x, x), n_terms=32))))(sweep)
real_grad = jax.vmap(jax.grad(lambda x: jnp.squeeze(di.midpoint(dirichlet.dirichlet_zeta(di.interval(x, x), n_terms=32)))))(sweep)
complex_primal = jax.vmap(lambda x: jnp.real(acb_core.acb_midpoint(acb_dirichlet.acb_dirichlet_zeta(acb_core.acb_box(di.interval(x, x), di.interval(0.2, 0.2)), n_terms=48))))(sweep)
complex_grad = jax.vmap(jax.grad(lambda x: jnp.real(acb_core.acb_midpoint(acb_dirichlet.acb_dirichlet_zeta(acb_core.acb_box(di.interval(x, x), di.interval(0.2, 0.2)), n_terms=48))))))(sweep)
ad_df = pd.DataFrame({'s': [float(v) for v in sweep], 'real_primal': [float(v) for v in real_primal], 'real_grad': [float(v) for v in real_grad], 'complex_primal': [float(v) for v in complex_primal], 'complex_grad': [float(v) for v in complex_grad]})
display(ad_df.head())
ax = ad_df.plot(x='s', y=['real_primal', 'real_grad', 'complex_primal', 'complex_grad'], figsize=(10, 4), title='Dirichlet AD Validation')
ax.set_ylabel('value')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Validation Summary

Run the Dirichlet chassis and engineering tests that back the real and complex helper surfaces.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_dirichlet_chassis.py',
    'tests/test_acb_dirichlet_chassis.py',
    'tests/test_dirichlet_engineering.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
(EXAMPLE_OUTPUT_ROOT / f'pytest_{JAX_MODE}.txt').write_text(tests.stdout + ('\n' + tests.stderr if tests.stderr else ''), encoding='utf-8')

## Benchmark Summary

Run the real and complex Dirichlet benchmark entrypoints and summarize their reported warm timings.

In [ ]:
real_report = EXAMPLE_OUTPUT_ROOT / f'dirichlet_real_{JAX_MODE}.json'
complex_report = EXAMPLE_OUTPUT_ROOT / f'dirichlet_complex_{JAX_MODE}.json'
run([PYTHON, 'benchmarks/benchmark_dirichlet.py', '--which', 'zeta', '--samples', '512', '--terms', '32', '--dtype', JAX_DTYPE, '--jax-mode', JAX_MODE, '--output', str(real_report)])
run([PYTHON, 'benchmarks/benchmark_acb_dirichlet.py', '--which', 'zeta', '--samples', '512', '--terms', '48', '--dtype', JAX_DTYPE, '--jax-mode', JAX_MODE, '--output', str(complex_report)])
bench_rows = []
for path in (real_report, complex_report):
    payload = json.loads(path.read_text())
    bench_rows.extend(payload['records'])
bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv(EXAMPLE_OUTPUT_ROOT / f'dirichlet_benchmark_summary_{JAX_MODE}.csv', index=False)
display(bench_df[['implementation', 'operation', 'dtype', 'warm_time_s']])

## Comparison Summary

Run the existing reference compare scripts when the local C reference libraries are available; otherwise record the missing-reference state explicitly.

In [ ]:
compare_cmds = [
    [PYTHON, 'benchmarks/compare_dirichlet.py', '--samples', '256', '--terms', '24', '--which', 'zeta'],
    [PYTHON, 'benchmarks/compare_acb_dirichlet.py', '--samples', '256', '--terms', '24'],
]
comparison_rows = []
for cmd in compare_cmds:
    completed = subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=True)
    comparison_rows.append({'script': cmd[1], 'returncode': completed.returncode, 'stdout': completed.stdout[-2000:], 'stderr': completed.stderr[-2000:]})
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
(EXAMPLE_OUTPUT_ROOT / f'comparison_status_{JAX_MODE}.json').write_text(json.dumps(comparison_rows, indent=2) + '\n', encoding='utf-8')
display(pd.DataFrame(comparison_rows)[['script', 'returncode']])

## Plots

Plot the benchmark summary and the real midpoint relation `eta = (1 - 2^(1-s)) zeta` over the representative sweep.

In [ ]:
eta_mid = jax.vmap(lambda x: jnp.squeeze(di.midpoint(dirichlet.dirichlet_eta(di.interval(x, x), n_terms=32))))(sweep)
relation_df = pd.DataFrame({'s': [float(v) for v in sweep], 'zeta_mid': [float(v) for v in real_primal], 'eta_mid': [float(v) for v in eta_mid]})
relation_df['eta_from_zeta'] = (1.0 - 2.0 ** (1.0 - relation_df['s'])) * relation_df['zeta_mid']
ax = bench_df.plot(x='implementation', y='warm_time_s', kind='bar', figsize=(8, 4), title='Dirichlet Warm Time Summary')
ax.set_ylabel('warm_time_s')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'dirichlet_benchmark_summary_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()
ax = relation_df.plot(x='s', y=['eta_mid', 'eta_from_zeta'], figsize=(8, 4), title='Dirichlet Eta Relation Check')
ax.set_ylabel('value')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'dirichlet_eta_relation_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()